In [ ]:
#################################################################################################
# Jag skapade en Lambda-funktion inne i AWS Lambda som använder Amazon Rekognition för att detektera etiketter i bilder som finns i en S3-bucket. 
# Alla bilder sparades i en S3-bucket som heter "aws-rekognition-picture-lables". 
# - Lambda-funktionen läser alla bilder i bucketen
# - Rekognition används för etikettigenkänning och samlar resultaten i en lista.
# Resultaten sparades sedan i en JSON-fil i AWS S3.
# Se bilder som använts och laddats upp i AWS S3 här i mappen Amazon_Rekognition/Pictures, 54st.
# Se fil "aws_lambda_label-detection.py" för koden som används i AWS Lambda.
# Se fil "rekognition_resultat.json" för resultatet av detekteringen i JSON-format.
# Se fil "aws_analysera_rekognition.py" för att analys av resultatet av detekteringen.
# Testade först MinConfidence=90 men träffsäkerheten blev 40.7% vilket inte blev så bra. 10 bilder saknade label helt, 22 felklassade och 22 rättklassade bilder.
# Sänkte till MinConfidence=70 och då blev träffsäkerheten 61.1% vilket är mycket bättre. 3 bilder saknade label, 18 felklassade och 33 rättklassade bilder. 
# Mycket bättre resultat med 70% än 90% och det är intressant att se att det inte blev så många fler felklassade bilder, utan att det istället blev fler rättklassade bilder. 
# Det visar att det kan vara värt att sänka MinConfidence för att få bättre träffsäkerhet, även om det kan leda till fler felklassningar. 
# Det är viktigt att hitta en balans mellan träffsäkerhet och antal klassningar för att få bästa möjliga resultat.

# Jag skapade en ny bucket i S3: test2pictures och laddade upp 19 nya testbilder från Kaggle. Gjorde om truelabels i Lambda-funktionen och körde om detekteringen. 
# Träffsäkerheten blev 84.2%% vilket är mycket bättre. Här hittar Rekognition rätt label i 16 av 19 bilder, och i 3 av bilderna saknas label helt. Det blev inga felklassningar.
# Dom 3 felträffarna är egentligen inte felträffar utan att true label inte visar ut allt som fanns i bilden. Ex visar Amazon Rekognition "Shoe" men true label var "Market". på bilden fanns det tydligt skor, så det är också rätt. 
# Min slutsat är att AWS Rekognition gör ett bra jobb med att detektera etiketter i generella bilder och hittar olika saker inom varje bild. 
# Är det ett foto med en sak i så kan det vara svårt att få en träffsäkerhet på 100%. 

# Amazon Rekognition verkar inte fungera bra på vardagsbilder, som i mitt fall med vit bakgrund. Dom verkar oftare ha bredare kategorier som "Animal" eller "Mammal" än mer specifika kategorier som "Dog" eller "Cat". 
#  Det är intressant att se att det inte blev så många fler felklassade bilder när jag sänkte MinConfidence från 90% till 70%, utan att det istället blev fler rättklassade bilder.
# Det visar att det kan vara värt att sänka MinConfidence för att få bättre träffsäkerhet, även om det can leda till fler felklassningar.
# #################################################################################################



import json
import boto3

client = boto3.client('rekognition')
s3 = boto3.client('s3')

true_labels = {
    "IMG_2192.JPG": ["Straw", "Drinking Straw"],
    "IMG_2193.JPG": ["Battery"],
    "IMG_2194.JPG": ["Glasses Case", "Eyeglasses Case"],
    "IMG_2195.JPG": ["Hooks", "Metal Hooks"],
    "IMG_2196.JPG": ["Pens", "Pen"],
    "IMG_2197.JPG": ["Nail File"],
    "IMG_2198.JPG": ["Headphones"],
    "IMG_2199.JPG": ["Battery String Lights", "String Lights"],
    "IMG_2200.JPG": ["Glove"],
    "IMG_2201.JPG": ["Game Controller"],
    "IMG_2202.JPG": ["Glasses", "Sunglasses", "Accessories"],
    "IMG_2203.JPG": ["Remote Control"],
    "IMG_2204.JPG": ["Tealight", "Candle Holder"],
    "IMG_2205.JPG": ["Toy", "Stuffed Toy"],
    "IMG_2206.JPG": ["Ruler"],
    "IMG_2207.JPG": ["Scissors", "shears"],
    "IMG_2208.JPG": ["Toy Money", "Banknote", "Circuit Board"],
    "IMG_2209.JPG": ["Roll-On", "Deodorant"],
    "IMG_2210.JPG": ["Eraser", "Rubber Eraser"],
    "IMG_2211.JPG": ["Rubik's Cube", "Toy", "Box"],
    "IMG_2212.JPG": ["Earbuds", "Headphones"],
    "IMG_2213.JPG": ["Keychain"],
    "IMG_2214.JPG": ["Medal", "Gold Medal"],
    "IMG_2215.JPG": ["Toy", "Toy Figures"],
    "IMG_2216.JPG": ["Cat", "Porcelain Cat", "Ornament Cat"],
    "IMG_2217.JPG": ["Dog", "Porcelain Dog", "Ornament Dog"],
    "IMG_2218.JPG": ["Soccer Ball", "Football", "Ball"],
    "IMG_2219.JPG": ["Coin"],
    "IMG_2220.JPG": ["Bowl", "Snow Globe", "Figurine"],
    "IMG_2221.JPG": ["Bottle", "Prime Bottle"],
    "IMG_2222.JPG": ["Charger", "Adapter"],
    "IMG_2223.JPG": ["Cable"],
    "IMG_2224.JPG": ["Scrunchie", "Hair Tie"],
    "IMG_2225.JPG": ["Watch", "Wristwatch", "Jewelry"],
    "IMG_2226.JPG": ["Tape"],
    "IMG_2227.JPG": ["Keychain", "Lucky Charm", "Evil Eye", "Horseshoe"],
    "IMG_2228.JPG": ["Light Bulb", "Lightbulb"],
    "IMG_2229.JPG": ["Lens Cap", "Camera Lens Cap"],
    "IMG_2230.JPG": ["Pegboard", "Bead Board"],
    "IMG_2231.JPG": ["Dala Horse", "Figurine"],
    "IMG_2232.JPG": ["Knife"],
    "IMG_2233.JPG": ["Fork"],
    "IMG_2234.JPG": ["Spoon"],
    "IMG_2235.JPG": ["Egg Slicer"],
    "IMG_2236.JPG": ["Potato Peeler", "Kitchen Peeler", "Peeler"],
    "IMG_2238.JPG": ["Stone"],
    "IMG_2239.JPG": ["Coaster", "Drink Coaster"],
    "IMG_2240.JPG": ["USB Flash Drive", "USB Stick"],
    "IMG_2241.JPG": ["Flower"],
    "IMG_2242.JPG": ["Chewing Gum Pack", "Gum Pack"],
    "IMG_2243.JPG": ["Speaker", "Computer Speaker"],
    "IMG_2244.JPG": ["Santa", "Beard", "Santa Hat", "Santa Decoration"],
    "IMG_2245.JPG": ["Bottle Opener"],
    "IMG_2246.JPG": ["Razor"]
}

def lambda_handler(event, context):
    bucket_name = "aws-rekognition-picture-lables"
    results = []

    s3_response = s3.list_objects_v2(Bucket=bucket_name)

    if "Contents" not in s3_response:
        return {
            'statusCode': 200,
            'body': json.dumps("Inga filer hittades i bucket.", ensure_ascii=False)
        }

    for obj in s3_response['Contents']:
        image_name = obj['Key']
        print(image_name)

        if image_name.startswith('resultat/'):
            continue

        if image_name.endswith('/'):
            continue

        if not image_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        rekognition_response = client.detect_labels(
            Image={
                'S3Object': {
                    'Bucket': bucket_name,
                    'Name': image_name
                }
            },
            MaxLabels=10,
            MinConfidence=70
        )

        labels = []
        for label in rekognition_response['Labels']:
            labels.append({
                'Name': label['Name'],
                'Confidence': round(label['Confidence'], 2),
                'Parents': [parent['Name'] for parent in label.get('Parents', [])]
            })

        file_name = image_name.split('/')[-1]

        results.append({
            'image': image_name,
            'file_name': file_name,
            'true_labels': true_labels.get(file_name, []),
            'labels': labels
        })

    json_data = json.dumps(results, ensure_ascii=False, indent=2)

    s3.put_object(
        Bucket=bucket_name,
        Key='resultat/rekognition_resultat.json',
        Body=json_data.encode('utf-8'),
        ContentType='application/json'
    )

    return {
        'statusCode': 200,
        'body': json.dumps({
            'message': 'Klart',
            'antal_bilder': len(results),
            'json_fil': 'resultat/rekognition_resultat.json'
        }, ensure_ascii=False)
    }

ModuleNotFoundError: No module named 'boto3'